In [0]:
order_silver = spark.table("02_silver_catalog.silver.order")
invoice_silver = spark.table("02_silver_catalog.silver.invoice")
estimate_silver = spark.table("02_silver_catalog.silver.estimate")
customer_survey_silver = spark.table("02_silver_catalog.silver.customer_survey")
store_silver = spark.table("02_silver_catalog.silver.store")
ns_budget_silver = spark.table("02_silver_catalog.silver.ns_budget")

In [0]:
gold_catalog = "03_gold_catalog.gold."

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### Dimensional tables

In [0]:
# Dimension table using natural key (store_id)
dim_store = (
    store_silver
    .select(
        "store_id",
        "store_name",
        "city",
        "state",
        "manager_id",
        "manager_name",
        "opened_year",
        "store_type"
    )
)

print(f"dim_store: {dim_store.count()} rows, {len(dim_store.columns)} columns")
display(dim_store.limit(5))

dim_store.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_store")

In [0]:
# Dimension table using natural key (technician_id)
dim_technician = (
    order_silver
    .filter(F.col("technician_id").isNotNull())
    .select("technician_id", "technician_name")
    .distinct()
    .dropDuplicates(["technician_id"])
    .orderBy("technician_id")
)

print(f"dim_technician: {dim_technician.count()} rows, {len(dim_technician.columns)} columns")
display(dim_technician.limit(5))

dim_technician.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_technician")

In [0]:
# Dimension table using natural key (estimator_id)
dim_estimator = (
    estimate_silver
    .filter(F.col("estimator_id").isNotNull())
    .select("estimator_id", "estimator_name")
    .distinct()
    .dropDuplicates(["estimator_id"])
)

print(f"dim_estimator: {dim_estimator.count()} rows, {len(dim_estimator.columns)} columns")

dim_estimator.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_estimator")

In [0]:
# Dimension table using natural key (vehicle_no)
# Remove duplicates by keeping first make/model per vehicle_no
dim_vehicle = (
    order_silver
    .select("vehicle_no", "vehicle_make", "vehicle_model")
    .dropDuplicates(["vehicle_no"])  # Keep first occurrence of each vehicle_no
)

print(f"dim_vehicle: {dim_vehicle.count()} rows (deduplicated)")

dim_vehicle.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_vehicle")

In [0]:
# Dimension table using natural key (date)
min_date = (
    order_silver
        .select(F.min(F.to_date("vehicle_in_datetime")).alias("min_date"))
        .first()[0]
)

max_date = (
    invoice_silver
        .select(F.max("invoice_date").alias("max_date"))
        .first()[0]
)

print("MIN DATE:", min_date)
print("MAX DATE:", max_date)

if min_date is None or max_date is None:
    raise ValueError("Cannot build dim_date because min_date or max_date is NULL.")

# Create date range
date_df = (
    spark.range(0, (max_date - min_date).days + 1)
         .withColumn("id_int", F.col("id").cast("int"))
         .withColumn("date", F.expr(f"date_add('{min_date}', id_int)"))
         .drop("id", "id_int")
)

# Build dim_date with natural key (date)
dim_date = (
    date_df
        .withColumn("date_key", F.date_format("date", "yyyyMMdd"))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
        .withColumn("day", F.dayofmonth("date"))
        .withColumn("quarter", F.quarter("date"))
        .withColumn("week_of_year", F.weekofyear("date"))
        .withColumn("day_of_week", F.date_format("date", "E"))
        .withColumn("is_weekend", F.col("day_of_week").isin("Sat", "Sun"))
)

print(f"dim_date: {dim_date.count()} rows")

dim_date.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_date")

In [0]:
# Dimension table using composite natural key (store_id, budget_year, budget_month)
dim_budget = (
    ns_budget_silver
    .select(
        F.col("ns_store_id").alias("store_id"),
        "budget_year",
        "budget_month",
        "budget_date",
        "budget_amount",
        "approved_by"
    )
)

print(f"dim_budget: {dim_budget.count()} rows, {len(dim_budget.columns)} columns")
display(dim_budget.limit(5))

dim_budget.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_budget")

### Fact Tables

In [0]:
# Enrich order data with estimate information
# Supports Use Case 8: Estimator Accuracy (initial vs final estimates)
# Uses natural keys only - no surrogate keys

# Step 1: Get initial and final estimates per order
window_spec = Window.partitionBy("order_id")

estimate_enriched = (
    estimate_silver
    .withColumn("min_version", F.min("version_no").over(window_spec))
    .withColumn("max_version", F.max("version_no").over(window_spec))
    .withColumn(
        "initial_estimate_amount",
        F.when(F.col("version_no") == F.col("min_version"), F.col("estimate_amount")).otherwise(None)
    )
    .withColumn(
        "final_estimate_amount",
        F.when(F.col("version_no") == F.col("max_version"), F.col("estimate_amount")).otherwise(None)
    )
    .groupBy("order_id")
    .agg(
        F.first("estimator_id", ignorenulls=True).alias("estimator_id"),
        F.max("initial_estimate_amount").alias("initial_estimate_amount"),
        F.max("final_estimate_amount").alias("final_estimate_amount")
    )
)

# Step 2: Join estimates to orders (keep all order columns)
fact_order_cycle = (
    order_silver
    .join(estimate_enriched, "order_id", "left")
    .drop("technician_name", "vehicle_make", "vehicle_model", "customer_name", "customer_phone")
)

print(f"fact_order_cycle: {fact_order_cycle.count()} rows")
display(fact_order_cycle.limit(5))

fact_order_cycle.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"fact_order_cycle")

In [0]:
invoice_df = invoice_silver.alias("i")
order_df = order_silver.alias("o")

fact_invoice = (
    invoice_df
    .join(order_df, "order_id", "left")
    .select(
        "i.invoice_id",
        "i.order_id",
        "o.store_id",
        "o.technician_id",
        "invoice_amount",
        "payment_mode",
        "currency",
        "invoice_date",
        "invoice_ts"
    )
)

print(f"fact_invoice: {fact_invoice.count()} rows")

fact_invoice.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"fact_invoice")

In [0]:
survey_df = customer_survey_silver.alias("s")
order_df = order_silver.alias("o")

fact_survey = (
    survey_df
    .join(order_df, "order_id", "left")
    .select(
        "s.survey_id",
        "s.order_id",
        "o.store_id",
        "o.technician_id",
        "responded_flag",
        "survey_sent_date",
        "survey_response_date",
        "overall_satisfaction_rating",
        "cleanliness_rating",
        "communication_rating",
        "work_quality_rating",
        "delivered_on_time_rating",
        F.datediff("survey_response_date", "survey_sent_date").alias("days_to_respond")
    )
)

print(f"fact_survey: {fact_survey.count()} rows")

fact_survey.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"fact_survey")

### Join fact to Dim

In [0]:
# Join fact_order_cycle to dimensions using natural keys
# Denormalize with dimension attributes for query performance

fact_order = fact_order_cycle.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_estimator_df = dim_estimator.alias("de")
dim_vehicle_df = dim_vehicle.alias("dv")
dim_date_df = dim_date.alias("dd")

# Join using natural keys
fact_order_dim = (
    fact_order
    .join(dim_store_df, fact_order.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_order.technician_id == dim_technician_df.technician_id, "left")
    .join(dim_estimator_df, fact_order.estimator_id == dim_estimator_df.estimator_id, "left")
    .join(dim_vehicle_df, fact_order.vehicle_no == dim_vehicle_df.vehicle_no, "left")

    # Join date dimensions
    .join(dim_date_df.withColumnRenamed("date_key", "vehicle_in_date_key"),
          F.to_date(fact_order.vehicle_in_datetime) == dim_date_df.date, "left")

    .join(dim_date_df.alias("dd2").withColumnRenamed("date_key", "delivery_date_key"),
          fact_order.actual_delivery_datetime.cast("date") == F.col("dd2.date"), "left")
)

# Select final fields - natural keys + denormalized attributes
fact_order_dim = fact_order_dim.select(
    # Natural Keys
    "f.order_id",
    "f.store_id",
    "f.technician_id",
    "f.estimator_id",
    "f.vehicle_no",
    
    # Denormalized Dimension Attributes
    "ds.store_name",
    "ds.manager_name",
    "dt.technician_name",
    "de.estimator_name",
    "dv.vehicle_make",
    "dv.vehicle_model",
    
    # Date Keys
    "vehicle_in_date_key",
    "delivery_date_key",
    
    # Fact Measures
    "f.service_type",
    "f.order_status",
    "f.vehicle_in_datetime",
    "f.vehicle_out_datetime",
    "f.actual_work_start_datetime",
    "f.actual_completion_datetime",
    "f.actual_delivery_datetime",
    "f.vehicle_in_to_work_start_days",
    "f.work_start_to_completion_days",
    "f.work_completion_to_delivery_days",
    "f.days_in_shop",
    "f.planned_vs_actual_completion_days",
    "f.promised_vs_actual_delivery_days",
    
    # Estimate Measures
    "f.initial_estimate_amount",
    "f.final_estimate_amount"
)

print(f"fact_order_dim: {fact_order_dim.count()} rows")
display(fact_order_dim.limit(5))

# Write back to Gold Layer
fact_order_dim.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.fact_order_dim")

In [0]:
# Join fact_invoice to dimensions using natural keys
# Denormalize with dimension attributes

fact_inv = fact_invoice.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_date_df = dim_date.alias("dd")

fact_invoice_dim = (
    fact_inv
    .join(dim_store_df, fact_inv.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_inv.technician_id == dim_technician_df.technician_id, "left")
    .join(dim_date_df,
          fact_inv.invoice_date == dim_date_df.date, "left")
)

fact_invoice_conformed = fact_invoice_dim.select(
    # Natural Keys
    "f.invoice_id",
    "f.order_id",
    "f.store_id",
    "f.technician_id",
    
    # Denormalized Attributes
    "ds.store_name",
    "ds.manager_name",
    "dt.technician_name",
    
    # Date columns for Revenue vs Budget
    F.col("dd.date_key").alias("invoice_date_key"),
    F.col("dd.year").alias("invoice_year"),
    F.col("dd.month").alias("invoice_month"),
    
    # Fact Measures
    "f.invoice_amount",
    "f.payment_mode",
    "f.currency",
    "f.invoice_date",
    "f.invoice_ts"
)

print(f"fact_invoice_dim: {fact_invoice_conformed.count()} rows")

fact_invoice_conformed.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.fact_invoice_dim")

In [0]:
# Join fact_survey to dimensions using natural keys
# Denormalize with dimension attributes

fact_s = fact_survey.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_date_df = dim_date.alias("dd")

fact_survey_dim = (
    fact_s
    .join(dim_store_df, fact_s.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_s.technician_id == dim_technician_df.technician_id, "left")

    .join(dim_date_df.withColumnRenamed("date_key", "survey_sent_date_key"),
          fact_s.survey_sent_date == dim_date_df.date, "left")

    .join(dim_date_df.alias("dd2").withColumnRenamed("date_key", "survey_response_date_key"),
          fact_s.survey_response_date == F.col("dd2.date"), "left")
)

fact_survey_conformed = fact_survey_dim.select(
    # Natural Keys
    "f.survey_id",
    "f.order_id",
    "f.store_id",
    "f.technician_id",
    
    # Denormalized Attributes
    "ds.store_name",
    "dt.technician_name",
    
    # Date Keys
    "survey_sent_date_key",
    "survey_response_date_key",
    
    # Fact Measures
    "f.responded_flag",
    "f.survey_sent_date",
    "f.survey_response_date",
    "f.overall_satisfaction_rating",
    "f.cleanliness_rating",
    "f.communication_rating",
    "f.work_quality_rating",
    "f.delivered_on_time_rating",
    "f.days_to_respond"
)

print(f"fact_survey_dim: {fact_survey_conformed.count()} rows")

fact_survey_conformed.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.fact_survey_dim")